# Bài 5.4(a): Trục của ellipsoid tin cậy 90%

Notebook này kiểm tra lại các phép tính trong lời giải: phương trình đặc trưng, trị riêng, vectơ riêng chuẩn hóa và độ dài các trục của ellipsoid tin cậy.

In [22]:
from pathlib import Path
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.precision", 4)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

## 1. Dữ kiện từ Ví dụ 5.2

Với dữ liệu mồ hôi trong Bảng 5.1, sách cho $n=20$, $p=3$, vector trung bình mẫu và ma trận hiệp phương sai mẫu như sau.

In [23]:
n = 20
p = 3
alpha = 0.10
F_3_17_upper_010 = 2.44  # Gia tri sach dung cho F_{3,17}(0.10)
variables = ["Sweat rate", "Sodium", "Potassium"]

xbar = np.array([4.640, 45.400, 9.965])
S = np.array(
    [
        [2.879, 10.010, -1.810],
        [10.010, 199.788, -5.640],
        [-1.810, -5.640, 3.628],
    ]
)

display(pd.DataFrame(xbar, index=variables, columns=["xbar"]))
display(pd.DataFrame(S, index=variables, columns=variables))

,xbar
Sweat rate,4.640
Sodium,45.400
Potassium,9.965


,Sweat rate,Sodium,Potassium
Sweat rate,2.879,10.010,-1.810
Sodium,10.010,199.788,-5.640
Potassium,-1.810,-5.640,3.628


## 2. Phương trình đặc trưng

Trị riêng của $S$ là nghiệm của $|S-\lambda I|=0$. Đoạn code dưới đây tính các hệ số của đa thức đặc trưng.

In [24]:
poly_coeffs = np.poly(S)

pd.DataFrame(
    {
        "coefficient": poly_coeffs,
        "term": ["lambda^3", "lambda^2", "lambda", "constant"],
    }
)

,coefficient,term
0,1.0000,lambda^3
1,-206.2950,lambda^2
2,1175.1797,lambda
3,-1181.5290,constant


Vì vậy đa thức có dạng gần đúng


$$
\lambda^3 - 206.295\lambda^2 + 1175.180\lambda - 1181.529 = 0.
$$

## 3. Trị riêng và vectơ riêng chuẩn hóa

Vì $S$ đối xứng, dùng `np.linalg.eigh`. Hàm này trả về các vectơ riêng đã chuẩn hóa theo chuẩn Euclid. Ta sắp xếp trị riêng theo thứ tự giảm dần để khớp lời giải.

In [25]:
eigenvalues, eigenvectors = np.linalg.eigh(S)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Doi dau de huong vector khop voi cach trinh bay trong loi giai.
if eigenvectors[1, 0] < 0:
    eigenvectors[:, 0] *= -1
for col in [1, 2]:
    if eigenvectors[0, col] < 0:
        eigenvectors[:, col] *= -1

eig_table = pd.DataFrame(
    np.column_stack([eigenvalues, eigenvectors.T]),
    columns=["lambda", "e_1 component", "e_2 component", "e_3 component"],
    index=["axis 1", "axis 2", "axis 3"],
)
eig_table

,lambda,e_1 component,e_2 component,e_3 component
axis 1,200.4620,0.0508,0.9983,-0.0291
axis 2,4.5326,0.5737,-0.0530,-0.8173
axis 3,1.3004,0.8175,-0.0249,0.5754


Để trình bày vectơ riêng chưa chuẩn hóa, ta có thể đặt một thành phần thuận tiện bằng 1 rồi lấy các thành phần còn lại theo tỉ lệ. Vectơ riêng chỉ xác định đến một hằng số nhân khác 0, nên cách chọn này không làm đổi hướng trục.

In [26]:
chosen_component = [1, 0, 0]  # axis 1 dat thanh phan Sodium = 1; axis 2,3 dat thanh phan Sweat rate = 1
raw_vectors = []

for i, component in enumerate(chosen_component):
    raw = eigenvectors[:, i] / eigenvectors[component, i]
    raw_vectors.append(raw)

raw_vectors = np.column_stack(raw_vectors)

pd.DataFrame(
    {
        "lambda": eigenvalues,
        "v_i^T before normalization": [np.round(raw_vectors[:, i], 4).tolist() for i in range(3)],
        "norm(v_i)": [np.linalg.norm(raw_vectors[:, i]) for i in range(3)],
        "e_i^T after normalization": [np.round(eigenvectors[:, i], 4).tolist() for i in range(3)],
    },
    index=["i=1", "i=2", "i=3"],
)

,lambda,v_i^T before normalization,norm(v_i),e_i^T after normalization
i=1,200.4620,"[0.0509, 1.0, -0.0291]",1.0017,"[0.0508, 0.9983, -0.0291]"
i=2,4.5326,"[1.0, -0.0924, -1.4247]",1.7431,"[0.5737, -0.053, -0.8173]"
i=3,1.3004,"[1.0, -0.0304, 0.7039]",1.2233,"[0.8175, -0.0249, 0.5754]"


Kiểm tra lại $S e_i \approx \lambda_i e_i$.

In [27]:
residuals = [
    np.linalg.norm(S @ eigenvectors[:, i] - eigenvalues[i] * eigenvectors[:, i])
    for i in range(3)
]

pd.DataFrame({"residual norm": residuals}, index=["i=1", "i=2", "i=3"])

,residual norm
i=1,2.8602e-14
i=2,8.6073e-15
i=3,5.7677e-15


## 4. Độ dài các trục của ellipsoid

Với ellipsoid tin cậy 90%, dùng $\alpha=0.10$ và giá trị trong sách $F_{3,17}(0.10)\approx 2.44$.

In [28]:
c2 = p * (n - 1) / (n - p) * F_3_17_upper_010
axis_factor = np.sqrt(c2 / n)
semi_axis_lengths = np.sqrt(eigenvalues) * axis_factor
full_axis_lengths = 2 * semi_axis_lengths

pd.DataFrame(
    {
        "lambda": eigenvalues,
        "semi-axis length": semi_axis_lengths,
        "full axis length": full_axis_lengths,
        "direction e_i": [np.round(eigenvectors[:, i], 4).tolist() for i in range(3)],
    },
    index=["axis 1", "axis 2", "axis 3"],
)

,lambda,semi-axis length,full axis length,direction e_i
axis 1,200.4620,9.0554,18.1109,"[0.0508, 0.9983, -0.0291]"
axis 2,4.5326,1.3617,2.7233,"[0.5737, -0.053, -0.8173]"
axis 3,1.3004,0.7293,1.4587,"[0.8175, -0.0249, 0.5754]"
